# RF-DETR → ExecuTorch Export & Inference

Export an RF-DETR detector to an **ExecuTorch program** (`.pte`) and run it on-device with the
ExecuTorch runtime — no GPU required.

ExecuTorch is PyTorch's on-device inference runtime for mobile and edge hardware. Unlike ONNX or TFLite,
the model is captured directly via `torch.export` (no intermediate conversion) and lowered to a hardware
backend:

| Backend | Target | Precision |
|---------|--------|-----------|
| **`xnnpack`** (this notebook) | portable CPU (Android / iOS / Linux / macOS) | fp32 |
| `coreml` | Apple Neural Engine | fp16 |
| `qnn` | Qualcomm Snapdragon HTP | fp16 |

`xnnpack` is the portable, pip-installable default and is validated to match eager PyTorch to ~1e-5.

## 1. Install

The `[executorch]` extra provides the exporter and the XNNPACK backend.

Colab ships mutually inconsistent preinstalled packages that otherwise crash `import rfdetr`, so two are
aligned: `torchaudio` is uninstalled (RF-DETR never uses it, but `transformers` imports it when present and
a `torch`/`torchaudio` CUDA-version mismatch then errors), and `pillow` is force-reinstalled to a clean
version (a half-upgraded PIL breaks `torchvision`'s import with `cannot import name '_Ink'`).

> **torch/executorch ABI pin** — loading a `.pte` via `executorch.runtime` needs a `torch` whose ABI
> matches the `executorch` wheel. For `executorch==1.3.1`, pin `torch<2.13`; a newer torch can silently
> break the runtime with an `undefined symbol` error at import. Export itself is unaffected — only the
> in-process runtime used here for inference.

> **Colab**: after this cell, **Runtime → Restart session**, then run from the next cell — Colab keeps the
> old package versions loaded until a restart.

In [ ]:
!pip install -q "rfdetr[executorch]>=1.9.0" "torch<2.13" supervision
!pip install -q --force-reinstall --no-deps "pillow==11.3.0"
!pip uninstall -q -y torchaudio

## 2. Setup

ExecuTorch export and XNNPACK inference are CPU-only — no GPU is needed.

In [ ]:
from pathlib import Path

import numpy as np
import torch

EXPORT_DIR = Path("export_executorch")
EXPORT_DIR.mkdir(exist_ok=True)
CONFIDENCE_THRESHOLD = 0.5

## 3. Sample image

A single street scene with several COCO classes (dog, bicycle, car) verifies the exported program
produces correct detections.

In [ ]:
import urllib.request

from PIL import Image

IMAGE_URL = "https://media.roboflow.com/notebooks/examples/dog.jpeg"
IMAGE_PATH = EXPORT_DIR / "sample.jpg"
if not IMAGE_PATH.exists():
    urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)

image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Sample image: {image.size[0]}×{image.size[1]}")

## 4. Export to a `.pte` program

`format="executorch"` requires an explicit `backend`. The COCO-pretrained `RFDETRSmall` is exported
directly — pass `pretrain_weights="<path/to/checkpoint.pth>"` to export a fine-tuned model instead.

The file is named after the model variant (`rfdetr-small.pte`). `torch.export` bakes a fixed input shape
into the graph (square at the model's resolution), so `dynamic_batch` is not supported — export one `.pte`
per batch size.

In [ ]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()
resolution = model.model_config.resolution  # square input the graph is traced with

pte_path = model.export(format="executorch", backend="xnnpack", output_dir=str(EXPORT_DIR))
print(f"ExecuTorch program: {pte_path}  ({pte_path.stat().st_size / 1e6:.1f} MB)")
print(f"Traced input resolution: {resolution}×{resolution}")

## 5. Load the program

`Runtime.get()` returns the process-wide ExecuTorch runtime; `load_program(...).load_method("forward")`
gives a callable graph. (If this import fails with an `undefined symbol` / `dlopen` error, the torch/
executorch ABIs are mismatched — reinstall with `torch<2.13`; see the install note above.)

In [ ]:
from executorch.runtime import Runtime

runtime = Runtime.get()
method = runtime.load_program(str(pte_path)).load_method("forward")

## 6. Run inference

The `.pte` expects the **same input the model was trained on**: NCHW, ImageNet-normalized, at the traced
resolution. `infer_transforms` builds that exact preprocessing pipeline.

`method.execute([input])` returns the two model outputs in order: `dets` (boxes, `cxcywh`, normalized)
and `labels` (class logits). `post_process` applies sigmoid + top-k, converts to `xyxy`, and rescales the
boxes to the original image size.

In [ ]:
from rfdetr.export.benchmark import infer_transforms, post_process

transforms = infer_transforms((resolution, resolution))
image_tensor, _ = transforms(image, None)

dets, labels = method.execute([image_tensor[None].float()])

orig_h, orig_w = image.height, image.width
target_sizes = torch.tensor([[orig_h, orig_w]])
result = post_process({"dets": dets, "labels": labels}, target_sizes)[0]
print(f"Raw detections returned by the program: {len(result['scores'])}")

## 7. Visualize with supervision

Wrap the post-processed boxes in a `supervision.Detections`, filter by confidence, and annotate the
original image with COCO class names.

In [ ]:
import supervision as sv

from rfdetr.assets.coco_classes import COCO_CLASSES

detections = sv.Detections(
    xyxy=result["boxes"].cpu().numpy(),
    confidence=result["scores"].cpu().numpy(),
    class_id=result["labels"].cpu().numpy().astype(int),
)
detections = detections[detections.confidence > CONFIDENCE_THRESHOLD]

# COCO-pretrained models emit sparse COCO category IDs (1–90) as class_id — look names up in the
# COCO_CLASSES {id: name} dict, not a 0-based list. (Fine-tuned models use 0-based class_names.)
labels_text = [
    f"{COCO_CLASSES.get(int(c), str(int(c)))} {conf:.2f}" for c, conf in zip(detections.class_id, detections.confidence)
]
print(f"Kept {len(detections)} detections above {CONFIDENCE_THRESHOLD}: {labels_text}")

annotated = sv.BoxAnnotator(thickness=3).annotate(scene=np.array(image).copy(), detections=detections)
annotated = sv.LabelAnnotator(text_scale=0.6, text_thickness=1, text_padding=4).annotate(
    scene=annotated, detections=detections, labels=labels_text
)

OUTPUT_PATH = EXPORT_DIR / "annotated_executorch.jpg"
Image.fromarray(annotated).save(OUTPUT_PATH)
print(f"Saved annotated image: {OUTPUT_PATH}")
sv.plot_image(annotated)

## Next steps

- **Deploy on-device** — copy the `.pte` to your Android / iOS / edge app and run it with the ExecuTorch
  runtime for that platform. See the [ExecuTorch docs](https://pytorch.org/executorch/).
- **Apple devices** — export with `backend="coreml"` (requires `pip install coremltools`) to run fp16 on
  the Neural Engine. Detections stay correct; raw tensor values differ at fp16 precision.
- **Qualcomm Snapdragon** — export with `backend="qnn", soc="SM8650"` (requires an ExecuTorch source build
  against the QAIRT SDK — not available via pip).
- **Fine-tuned weights** — pass `pretrain_weights="<path/to/checkpoint.pth>"` when constructing the model.
- See the [Export documentation](https://rfdetr.roboflow.com/learn/export/) for all formats and options.